In [0]:
dbutils.widgets.text("environment", "dev", "Environment")

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError("Environment must be either 'dev' or 'prod'.")

config = {
    "dev": {
        "catalogs": [
            "saleslt_dev",
            "salesjson_dev",
            "salescsv_dev"
        ],
        "external_locations": {
            "landing": "ext_landing_dev",
            "lakehouse": "ext_lakehouse_dev",
            "streaming": "ext_streaming_dev"
        },
        "etl_service_principal_application_id":
            "acc15410-5c5f-473e-bc6f-61b7946176a2"
    },

    "prod": {
        "catalogs": [
            "saleslt_prod",
            "salesjson_prod",
            "salescsv_prod"
        ],
        "external_locations": {
            "landing": "ext_landing_prod",
            "lakehouse": "ext_lakehouse_prod",
            "streaming": "ext_streaming_prod"
        },
        "etl_service_principal_application_id":
            "acc15410-5c5f-473e-bc6f-61b7946176a2"
    }
}

env = config[environment]

developers_group = "grp-dbx-developers"
analysts_group = "grp-dbx-analysts"

sp_application_id = env["etl_service_principal_application_id"]

print(f"Environment: {environment}")
print(f"Developers group: {developers_group}")
print(f"Analysts group: {analysts_group}")
print(f"ETL Service Principal: {sp_application_id}")

In [0]:
# COMMAND ----------

for catalog in env["catalogs"]:

    spark.sql(f"""
        GRANT USE CATALOG
        ON CATALOG `{catalog}`
        TO `{developers_group}`
    """)

    for schema in ["bronze", "silver", "gold"]:

        spark.sql(f"""
            GRANT USE SCHEMA
            ON SCHEMA `{catalog}`.`{schema}`
            TO `{developers_group}`
        """)

        if environment == "dev":

            spark.sql(f"""
                GRANT CREATE TABLE
                ON SCHEMA `{catalog}`.`{schema}`
                TO `{developers_group}`
            """)

            spark.sql(f"""
                GRANT SELECT, MODIFY
                ON SCHEMA `{catalog}`.`{schema}`
                TO `{developers_group}`
            """)

        else:

            spark.sql(f"""
                GRANT SELECT
                ON SCHEMA `{catalog}`.`{schema}`
                TO `{developers_group}`
            """)

In [0]:
for catalog in env["catalogs"]:

    spark.sql(f"""
        GRANT USE CATALOG
        ON CATALOG `{catalog}`
        TO `{analysts_group}`
    """)

    spark.sql(f"""
        GRANT USE SCHEMA
        ON SCHEMA `{catalog}`.`gold`
        TO `{analysts_group}`
    """)

    spark.sql(f"""
        GRANT SELECT
        ON SCHEMA `{catalog}`.`gold`
        TO `{analysts_group}`
    """)

In [0]:

for catalog in env["catalogs"]:

    spark.sql(f"""
        GRANT USE CATALOG
        ON CATALOG `{catalog}`
        TO `{sp_application_id}`
    """)

    for schema in ["bronze", "silver", "gold"]:

        spark.sql(f"""
            GRANT USE SCHEMA
            ON SCHEMA `{catalog}`.`{schema}`
            TO `{sp_application_id}`
        """)

        spark.sql(f"""
            GRANT CREATE TABLE
            ON SCHEMA `{catalog}`.`{schema}`
            TO `{sp_application_id}`
        """)

        spark.sql(f"""
            GRANT SELECT, MODIFY
            ON SCHEMA `{catalog}`.`{schema}`
            TO `{sp_application_id}`
        """)

print("ETL Service Principal grants completed.")

In [0]:
landing_location = env["external_locations"]["landing"]

spark.sql(f"""
    GRANT READ FILES
    ON EXTERNAL LOCATION `{landing_location}`
    TO `{sp_application_id}`
""")

if environment == "dev":

    spark.sql(f"""
        GRANT READ FILES
        ON EXTERNAL LOCATION `{landing_location}`
        TO `{developers_group}`
    """)

print("Landing external location grants completed.")

In [0]:

lakehouse_location = env["external_locations"]["lakehouse"]

spark.sql(f"""
    GRANT CREATE EXTERNAL TABLE
    ON EXTERNAL LOCATION `{lakehouse_location}`
    TO `{sp_application_id}`
""")

if environment == "dev":

    spark.sql(f"""
        GRANT CREATE EXTERNAL TABLE
        ON EXTERNAL LOCATION `{lakehouse_location}`
        TO `{developers_group}`
    """)

print("Lakehouse external location grants completed.")

In [0]:
streaming_location = env["external_locations"]["streaming"]

spark.sql(f"""
    GRANT READ FILES, WRITE FILES
    ON EXTERNAL LOCATION `{streaming_location}`
    TO `{sp_application_id}`
""")

if environment == "dev":

    spark.sql(f"""
        GRANT READ FILES, WRITE FILES
        ON EXTERNAL LOCATION `{streaming_location}`
        TO `{developers_group}`
    """)

print("Streaming external location grants completed.")

In [0]:
print("====================================================")
print("UNITY CATALOG SECURITY CONFIGURATION")
print("====================================================")
print(f"Environment: {environment}")
print(f"ETL Service Principal: {sp_application_id}")
print("====================================================")

# Catalog permissions validation
for catalog in env["catalogs"]:

    print(f"\n===== CATALOG: {catalog} =====")

    display(
        spark.sql(
            f"SHOW GRANTS ON CATALOG `{catalog}`"
        )
    )

# Schema permissions validation

for catalog in env["catalogs"]:

    for schema in ["bronze", "silver", "gold"]:

        print(f"\n===== {catalog}.{schema} =====")

        display(
            spark.sql(
                f"""
                SHOW GRANTS ON SCHEMA
                `{catalog}`.`{schema}`
                """
            )
        )

# External location permissions validation

for location in env["external_locations"].values():

    print(f"\n===== EXTERNAL LOCATION: {location} =====")

    display(
        spark.sql(
            f"""
            SHOW GRANTS ON EXTERNAL LOCATION `{location}`
            """
        )
    )



